# Financial Fraud Detection — PaySim
Full pipeline: load real data from Kaggle → EDA → feature engineering (including chain accounts + balance-error features) → train/val/test split → Logistic Regression baseline → XGBoost main model → evaluation with confusion matrix, precision/recall/F1/ROC-AUC/PR-AUC.

Run cells top to bottom with **Shift+Enter**.

In [ ]:
!pip install kagglehub xgboost imbalanced-learn scikit-learn pandas matplotlib seaborn -q

## 1. Load the real PaySim dataset from Kaggle

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os

path = kagglehub.dataset_download("ealaxi/paysim1")
print("Dataset downloaded to:", path)

csv_file = [f for f in os.listdir(path) if f.endswith(".csv")][0]
df = pd.read_csv(os.path.join(path, csv_file))
print("Shape:", df.shape)
df.head(10)

## 2. Exploratory Data Analysis (EDA)
Check the fraud rate, fraud by transaction type, and confirm the merchant-balance pattern before touching any model.

In [ ]:
print("Fraud rate:")
print(df['isFraud'].value_counts(normalize=True))
print()
print("Total fraud rows:", df['isFraud'].sum())
print()
print("Fraud rate by transaction type:")
print(df.groupby('type')['isFraud'].mean().sort_values(ascending=False))
print()
print("Nulls:", df.isnull().sum().sum())

In [ ]:
# Confirm: merchant destinations always have zero balance
merchant_df = df[df['nameDest'].str.startswith('M')]
print("Merchant-destination transactions:", len(merchant_df))
print("Fraud among merchant-destination transactions:", merchant_df['isFraud'].sum())
print("All zero balances for merchants?",
      (merchant_df['oldbalanceDest'] == 0).all() and (merchant_df['newbalanceDest'] == 0).all())

## 3. Feature Engineering — on the FULL dataset

Important: we do **not** filter down to a "clean" subset. Instead we turn the balance
inconsistencies themselves into features, so we keep all fraud cases in the data.


In [ ]:
# --- Drop leaky / non-predictive columns ---
# isFlaggedFraud = existing rule-based system's own guess -> leakage if used as input
df_model = df.drop(columns=['isFlaggedFraud'])

# --- Balance-error features (computed on ALL rows, not used to filter anything) ---
df_model['errorBalanceOrig'] = df_model['newbalanceOrig'] - df_model['oldbalanceOrg'] + df_model['amount']
df_model['errorBalanceDest'] = df_model['oldbalanceDest'] - df_model['newbalanceDest'] + df_model['amount']

# --- Simple derived balance features ---
df_model['orig_balance_diff'] = df_model['oldbalanceOrg'] - df_model['newbalanceOrig']
df_model['dest_balance_diff'] = df_model['newbalanceDest'] - df_model['oldbalanceDest']
df_model['orig_emptied'] = (df_model['newbalanceOrig'] == 0).astype(int)
df_model['amount_to_balance_ratio'] = df_model['amount'] / (df_model['oldbalanceOrg'] + 1)

# --- Chain accounts: accounts seen as BOTH a sender and a receiver anywhere in the data ---
all_orig = set(df_model['nameOrig'].unique())
all_dest = set(df_model['nameDest'].unique())
chain_accounts = all_orig & all_dest
print("Chain accounts found:", len(chain_accounts))

df_model['orig_is_chain'] = df_model['nameOrig'].isin(chain_accounts).astype(int)
df_model['dest_is_chain'] = df_model['nameDest'].isin(chain_accounts).astype(int)

# --- Graph-style aggregate features: how active is this sender overall? ---
sender_counts = df_model['nameOrig'].value_counts()
df_model['sender_txn_count'] = df_model['nameOrig'].map(sender_counts)

dest_counts = df_model['nameDest'].value_counts()
df_model['dest_txn_count'] = df_model['nameDest'].map(dest_counts)

print("Feature engineering done. New shape:", df_model.shape)

In [ ]:
# --- Encode transaction type, drop ID columns now that chain-account features are built ---
df_model = pd.get_dummies(df_model, columns=['type'], drop_first=True)
df_model = df_model.drop(columns=['nameOrig', 'nameDest'])

print(df_model.columns.tolist())

## 4. Train / Validation / Test split (stratified — preserves fraud ratio in every split)

In [ ]:
from sklearn.model_selection import train_test_split

X = df_model.drop(columns=['isFraud'])
y = df_model['isFraud']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f"Train: {X_train.shape[0]:,} rows | fraud: {y_train.sum():,}")
print(f"Val:   {X_val.shape[0]:,} rows | fraud: {y_val.sum():,}")
print(f"Test:  {X_test.shape[0]:,} rows | fraud: {y_test.sum():,}")

## 5. Baseline model — Logistic Regression
Simple, fast, interpretable comparison point. Uses `class_weight='balanced'` to handle the imbalance.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, roc_curve, classification_report
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

logreg = LogisticRegression(class_weight='balanced', max_iter=1000)
logreg.fit(X_train_scaled, y_train)

logreg_probs = logreg.predict_proba(X_val_scaled)[:, 1]
logreg_preds = (logreg_probs >= 0.5).astype(int)

print("=== Logistic Regression (baseline) ===")
print(f"Precision: {precision_score(y_val, logreg_preds):.3f}")
print(f"Recall:    {recall_score(y_val, logreg_preds):.3f}")
print(f"F1:        {f1_score(y_val, logreg_preds):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_val, logreg_probs):.3f}")
print(f"PR-AUC:    {average_precision_score(y_val, logreg_probs):.3f}")

## 6. Main model — XGBoost
Uses `scale_pos_weight` to make the model penalize missed fraud much more heavily,
without touching or duplicating the data (no SMOTE needed here — see note below).

In [ ]:
from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.1f}")

xgb = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
)
xgb.fit(X_train, y_train)

xgb_probs = xgb.predict_proba(X_val)[:, 1]
xgb_preds = (xgb_probs >= 0.5).astype(int)

print("=== XGBoost (main model) ===")
print(f"Precision: {precision_score(y_val, xgb_preds):.3f}")
print(f"Recall:    {recall_score(y_val, xgb_preds):.3f}")
print(f"F1:        {f1_score(y_val, xgb_preds):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_val, xgb_probs):.3f}")
print(f"PR-AUC:    {average_precision_score(y_val, xgb_probs):.3f}")
print()
print(classification_report(y_val, xgb_preds, target_names=['Legit', 'Fraud']))

### Optional: SMOTE (only if recall is still too low after the weighted XGBoost above)
Only ever fit SMOTE on the **training set** — never on validation/test, or your evaluation
numbers become artificially inflated and won't reflect real-world performance.

In [ ]:
# Uncomment to try SMOTE as an alternative/addition to scale_pos_weight
# from imblearn.over_sampling import SMOTE
#
# smote = SMOTE(random_state=42)
# X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
# print("Before SMOTE:", y_train.value_counts().to_dict())
# print("After SMOTE: ", y_train_sm.value_counts().to_dict())
#
# xgb_smote = XGBClassifier(eval_metric='logloss', n_estimators=300, max_depth=6,
#                            learning_rate=0.1, random_state=42, n_jobs=-1)
# xgb_smote.fit(X_train_sm, y_train_sm)
# smote_probs = xgb_smote.predict_proba(X_val)[:, 1]
# smote_preds = (smote_probs >= 0.5).astype(int)
# print("Precision:", precision_score(y_val, smote_preds))
# print("Recall:   ", recall_score(y_val, smote_preds))
# print("F1:       ", f1_score(y_val, smote_preds))

## 7. Evaluation plots — confusion matrix, ROC curve, feature importance

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_val, xgb_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Legit', 'Predicted Fraud'],
            yticklabels=['Actual Legit', 'Actual Fraud'])
plt.title('XGBoost — Confusion Matrix (Validation Set)')
plt.tight_layout()
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_val, xgb_probs)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f'XGBoost (AUC = {roc_auc_score(y_val, xgb_probs):.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(xgb.feature_importances_, index=X_train.columns).sort_values(ascending=False)
plt.figure(figsize=(7, 5))
importances.head(12).plot(kind='barh')
plt.gca().invert_yaxis()
plt.title('Top 12 Feature Importances (XGBoost)')
plt.tight_layout()
plt.show()

print(importances.head(10))

## 8. Final, untouched TEST SET evaluation
Only run this once, at the very end, after you've stopped tuning anything.

In [ ]:
test_probs = xgb.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= 0.5).astype(int)

print("=== FINAL TEST SET RESULTS ===")
print(f"Precision: {precision_score(y_test, test_preds):.3f}")
print(f"Recall:    {recall_score(y_test, test_preds):.3f}")
print(f"F1:        {f1_score(y_test, test_preds):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, test_probs):.3f}")
print(f"PR-AUC:    {average_precision_score(y_test, test_probs):.3f}")